# DINOv2 Classification Heads on DermaMNIST 28×28 (upscaled to 224 for DINOv2)
Full pipeline: feature extraction → train 5 heads → evaluate ACC + AUC → comparison plots.

**Runtime:** GPU (T4 recommended). Runtime → Change runtime type → T4 GPU.

In [ ]:
# Install dependencies
!pip install -q medmnist transformers timm scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
from pathlib import Path
import os, time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler, label_binarize
from medmnist import DermaMNIST, INFO
from torchvision import transforms
from transformers import AutoModel

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
N_CLASSES  = 7
BATCH_SIZE = 256

CLASS_NAMES = [v for k, v in sorted(INFO["dermamnist"]["label"].items(),
                                     key=lambda x: int(x[0]))]
CLASS_SHORT = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "VASC"]

FEAT_DIR  = Path("/content/features/cls")
PATCH_DIR = Path("/content/features/patch")
CKPT_DIR  = Path("/content/checkpoints")
for d in [FEAT_DIR, PATCH_DIR, CKPT_DIR]: d.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Classes: {CLASS_NAMES}")

## 1. Extract DINOv2 Features (28×28 → upscaled to 224)
Runs once and caches to `/content/features/`.

In [ ]:
dinov2_transform = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract(split):
    cls_path   = FEAT_DIR  / f"X_{split}.npy"
    patch_path = PATCH_DIR / f"X_patch_{split}.npy"
    y_path     = FEAT_DIR  / f"y_{split}.npy"

    if cls_path.exists() and patch_path.exists():
        print(f"[{split}] Loading cached...")
        return (np.load(cls_path), np.load(y_path),
                np.load(patch_path), np.load(y_path))

    print(f"[{split}] Extracting...")
    backbone = AutoModel.from_pretrained("facebook/dinov2-base").eval().to(DEVICE)
    dataset  = DermaMNIST(split=split, transform=dinov2_transform,
                          size=28, download=True)
    loader   = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)

    all_cls, all_patch, all_y = [], [], []
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            out     = backbone(pixel_values=imgs.to(DEVICE))
            cls     = out.last_hidden_state[:, 0, :].cpu().numpy()
            patches = out.last_hidden_state[:, 1:, :]   # (B, 256, 768)
            B       = patches.shape[0]
            patches = patches.permute(0,2,1).reshape(B,768,16,16).cpu().numpy()
            all_cls.append(cls.astype(np.float32))
            all_patch.append(patches.astype(np.float16))
            all_y.append(labels.numpy().flatten())
            if (i+1) % 10 == 0:
                print(f"  batch {i+1}/{len(loader)}")

    del backbone; torch.cuda.empty_cache()
    X_cls   = np.concatenate(all_cls)
    X_patch = np.concatenate(all_patch)
    y       = np.concatenate(all_y)
    np.save(cls_path, X_cls); np.save(patch_path, X_patch); np.save(y_path, y)
    np.save(PATCH_DIR / f"y_{split}.npy", y)
    print(f"  [{split}] CLS={X_cls.shape}  patch={X_patch.shape}")
    return X_cls, y, X_patch, y

X_tr_cls, y_tr, X_tr_patch, _ = extract("train")
X_va_cls, y_va, X_va_patch, _ = extract("val")
X_te_cls, y_te, X_te_patch, _ = extract("test")
print("Done.")

## 2. Build Data Loaders

In [ ]:
scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr_cls)
X_va_s   = scaler.transform(X_va_cls)
X_te_s   = scaler.transform(X_te_cls)

def cls_loader(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

def patch_loader(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X.astype(np.float32)),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=64, shuffle=shuffle)

def attn_loader(X, y, shuffle=False):
    Xt = torch.tensor(X.astype("float32")).permute(0,2,3,1).reshape(-1,256,768)
    ds = TensorDataset(Xt, torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=64, shuffle=shuffle)

tr_cls   = cls_loader(X_tr_s,     y_tr, shuffle=True)
va_cls   = cls_loader(X_va_s,     y_va)
te_cls   = cls_loader(X_te_s,     y_te)
tr_patch = patch_loader(X_tr_patch, y_tr, shuffle=True)
va_patch = patch_loader(X_va_patch, y_va)
te_patch = patch_loader(X_te_patch, y_te)
tr_attn  = attn_loader(X_tr_patch, y_tr, shuffle=True)
va_attn  = attn_loader(X_va_patch, y_va)
te_attn  = attn_loader(X_te_patch, y_te)

print(f"train={len(y_tr)}  val={len(y_va)}  test={len(y_te)}")

## 3. Model Definitions

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim=768, n=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),    nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, n))
    def forward(self, x): return self.net(x)

class ResBlock(nn.Module):
    def __init__(self, d, drop=0.3):
        super().__init__()
        self.b = nn.Sequential(nn.Linear(d,d), nn.BatchNorm1d(d), nn.ReLU(),
                                nn.Dropout(drop), nn.Linear(d,d), nn.BatchNorm1d(d))
        self.relu = nn.ReLU()
    def forward(self, x): return self.relu(x + self.b(x))

class ResidualMLP(nn.Module):
    def __init__(self, in_dim=768, h=512, n=7, drop=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim,h), nn.BatchNorm1d(h),
                                   nn.ReLU(), nn.Dropout(drop))
        self.r1 = ResBlock(h, drop); self.r2 = ResBlock(h, drop)
        self.head = nn.Linear(h, n)
    def forward(self, x): return self.head(self.r2(self.r1(self.proj(x))))

class CNNHead(nn.Module):
    def __init__(self, c=768, n=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256,128,3,padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,64,3,padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n))
    def forward(self, x): return self.net(x)

class CNNHeadV2(nn.Module):
    def __init__(self, c=768, n=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256,128,3,padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,64,3,padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, n))
    def forward(self, x): return self.net(x)

class AttnPoolHead(nn.Module):
    def __init__(self, pd=768, proj=256, heads=4, layers=2, n=7, drop=0.3):
        super().__init__()
        self.proj    = nn.Linear(pd, proj)
        self.pos_emb = nn.Embedding(256, proj)
        enc = nn.TransformerEncoderLayer(d_model=proj, nhead=heads,
              dim_feedforward=proj*4, dropout=drop, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=layers)
        self.attn  = nn.Linear(proj, 1)
        self.cls   = nn.Sequential(nn.LayerNorm(proj), nn.Dropout(drop),
                                    nn.Linear(proj,128), nn.GELU(),
                                    nn.Dropout(drop*0.5), nn.Linear(128, n))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        B, N, _ = x.shape
        pos = torch.arange(N, device=x.device).unsqueeze(0)
        h = self.proj(x) + self.pos_emb(pos)
        h = self.transformer(h)
        w = torch.softmax(self.attn(h), dim=1)
        return self.cls((w * h).sum(dim=1))

print("Model definitions ready.")

## 4. Training

In [ ]:
def train(model, tr_loader, va_loader, ckpt, epochs=100, lr=3e-4,
          label_smooth=0.0, scheduler="cosine", patience=None,
          warmup=0, clip=0.0, noise=0.0, name="model"):
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    if scheduler == "cosine":
        sch = CosineAnnealingLR(opt, T_max=epochs)
    else:
        sch = ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10, min_lr=1e-7)

    best, no_imp = 0.0, 0
    hist = {"tl": [], "vl": [], "va": []}

    for ep in range(1, epochs+1):
        # warmup
        if warmup and ep <= warmup:
            for pg in opt.param_groups: pg["lr"] = lr * ep / warmup

        model.train(); tl = 0.0
        for X_b, y_b in tr_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            if noise > 0: X_b = X_b + torch.randn_like(X_b) * noise
            opt.zero_grad(); loss = crit(model(X_b), y_b); loss.backward()
            if clip > 0: nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step(); tl += loss.item()
        tl /= len(tr_loader)

        model.eval(); vl, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_b, y_b in va_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                logits = model(X_b)
                vl += crit(logits, y_b).item()
                correct += (logits.argmax(1)==y_b).sum().item()
                total   += y_b.size(0)
        vl /= len(va_loader); va = correct/total

        if scheduler == "cosine": sch.step()
        elif ep > warmup:         sch.step(va)

        hist["tl"].append(tl); hist["vl"].append(vl); hist["va"].append(va)
        if va > best:
            best = va; torch.save(model.state_dict(), ckpt); no_imp=0; flag="✓"
        else:
            no_imp += 1; flag=""
        if patience and no_imp >= patience:
            print(f"  [{name}] Early stop ep {ep}"); break
        if ep % 10 == 0 or ep == 1:
            print(f"  [{name}] ep{ep:03d} | tr={tl:.4f} vl={vl:.4f} va={va:.4f} {flag}")

    print(f"  [{name}] best val acc: {best:.4f}")
    return hist, best

In [ ]:
histories = {}

# class weights for CNN v2
class_counts  = np.bincount(y_tr)
class_weights = torch.tensor(
    (1.0/class_counts) / (1.0/class_counts).sum() * N_CLASSES,
    dtype=torch.float32).to(DEVICE)

configs = [
    ("MLP",         MLPClassifier(),  te_cls,   tr_cls,  va_cls,
     dict(epochs=100, lr=3e-4, scheduler="cosine", name="MLP")),
    ("Residual MLP",ResidualMLP(),    te_cls,   tr_cls,  va_cls,
     dict(epochs=100, lr=3e-4, scheduler="cosine", name="ResMLP")),
    ("CNN Head",    CNNHead(),        te_patch, tr_patch,va_patch,
     dict(epochs=60,  lr=3e-4, label_smooth=0.1, scheduler="cosine", name="CNN")),
    ("CNN Head v2", CNNHeadV2(),      te_patch, tr_patch,va_patch,
     dict(epochs=200, lr=1e-4, label_smooth=0.1, scheduler="plateau",
          patience=30, warmup=5, name="CNN-v2")),
    ("Attn Pool",   AttnPoolHead(),   te_attn,  tr_attn, va_attn,
     dict(epochs=150, lr=3e-4, label_smooth=0.05, scheduler="plateau",
          patience=25, warmup=5, clip=1.0, noise=0.005, name="Attn")),
]

# Store test loaders for eval
test_loaders = {}
for name, model, te_loader, tr_loader, va_loader, kwargs in configs:
    print(f"\n{'='*50}\n[{name}]")
    model = model.to(DEVICE)
    ckpt  = CKPT_DIR / f"{name.replace(' ','_')}_best.pt"

    # CNN v2 needs weighted loss — handle separately
    if name == "CNN Head v2":
        opt2  = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-3)
        sch2  = ReduceLROnPlateau(opt2, mode="min", factor=0.5, patience=10, min_lr=1e-7)
        crit2 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        hist2 = {"tl":[], "vl":[], "va":[]}
        best2, no_imp2, WARMUP2, PAT2 = 0.0, 0, 5, 30
        for ep in range(1, 201):
            if ep <= WARMUP2:
                for pg in opt2.param_groups: pg["lr"] = 1e-4 * ep / WARMUP2
            model.train(); tl=0.0
            for X_b,y_b in tr_loader:
                X_b,y_b=X_b.to(DEVICE),y_b.to(DEVICE)
                opt2.zero_grad(); loss=crit2(model(X_b),y_b); loss.backward(); opt2.step()
                tl+=loss.item()
            tl/=len(tr_loader)
            model.eval(); vl,correct,total=0.0,0,0
            with torch.no_grad():
                for X_b,y_b in va_loader:
                    X_b,y_b=X_b.to(DEVICE),y_b.to(DEVICE)
                    logits=model(X_b); vl+=crit2(logits,y_b).item()
                    correct+=(logits.argmax(1)==y_b).sum().item(); total+=y_b.size(0)
            vl/=len(va_loader); va=correct/total; sch2.step(vl)
            hist2["tl"].append(tl); hist2["vl"].append(vl); hist2["va"].append(va)
            if va>best2: best2=va; torch.save(model.state_dict(),ckpt); no_imp2=0; flag="✓"
            else: no_imp2+=1; flag=""
            if no_imp2>=PAT2: print(f"  [CNN-v2] Early stop ep {ep}"); break
            if ep%10==0 or ep<=WARMUP2:
                print(f"  [CNN-v2] ep{ep:03d} | tr={tl:.4f} vl={vl:.4f} va={va:.4f} {flag}")
        print(f"  [CNN-v2] best val acc: {best2:.4f}")
        histories[name] = hist2
    else:
        h, _ = train(model, tr_loader, va_loader, ckpt, **kwargs)
        histories[name] = h

    test_loaders[name] = te_loader

print("\nAll training complete!")

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 7))
method_order = ["MLP","Residual MLP","CNN Head","CNN Head v2","Attn Pool"]

for col, name in enumerate(method_order):
    h  = histories[name]
    ep = range(1, len(h["tl"])+1)
    axes[0,col].plot(ep, h["tl"], label="train", color="#4C72B0")
    axes[0,col].plot(ep, h["vl"], label="val",   color="#DD8452")
    axes[0,col].set_title(f"{name} — Loss", fontsize=9)
    axes[0,col].legend(fontsize=7); axes[0,col].set_xlabel("Epoch")
    best_va = max(h["va"])
    axes[1,col].plot(ep, h["va"], color="#55A868")
    axes[1,col].axhline(best_va, color="red", linestyle="--",
                         label=f"best={best_va:.4f}", linewidth=1)
    axes[1,col].set_title(f"{name} — Val Acc", fontsize=9)
    axes[1,col].legend(fontsize=7); axes[1,col].set_xlabel("Epoch")

plt.suptitle("DINOv2 Classification Heads — Training Curves (DermaMNIST 28×28)", fontsize=12)
plt.tight_layout()
plt.savefig("/content/28_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Evaluation — ACC + AUC

In [ ]:
model_map = {
    "MLP":          MLPClassifier(),
    "Residual MLP": ResidualMLP(),
    "CNN Head":     CNNHead(),
    "CNN Head v2":  CNNHeadV2(),
    "Attn Pool":    AttnPoolHead(),
}

def evaluate(model, loader, ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    logits_all, y_all = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            logits_all.append(model(X_b.to(DEVICE)).cpu().numpy())
            y_all.extend(y_b.numpy())
    logits = np.concatenate(logits_all)
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_true = np.array(y_all)
    y_pred = logits.argmax(1)

    acc     = accuracy_score(y_true, y_pred)
    f1_mac  = f1_score(y_true, y_pred, average="macro")
    f1_wtd  = f1_score(y_true, y_pred, average="weighted")
    auc_mac = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    auc_wtd = roc_auc_score(y_true, probs, multi_class="ovr", average="weighted")
    y_bin   = label_binarize(y_true, classes=list(range(N_CLASSES)))
    per_auc = np.array([roc_auc_score(y_bin[:,i], probs[:,i]) for i in range(N_CLASSES)])
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    per_rec = cm_norm.diagonal()
    return {"acc":acc,"f1_mac":f1_mac,"f1_wtd":f1_wtd,
            "auc_mac":auc_mac,"auc_wtd":auc_wtd,
            "per_auc":per_auc,"per_rec":per_rec,"cm_norm":cm_norm}

results = {}
for name, model in model_map.items():
    model = model.to(DEVICE)
    ckpt  = CKPT_DIR / f"{name.replace(' ','_')}_best.pt"
    r = evaluate(model, test_loaders[name], ckpt)
    results[name] = r
    print(f"[{name}]  acc={r['acc']:.4f}  macro_auc={r['auc_mac']:.4f}  "
          f"wtd_auc={r['auc_wtd']:.4f}  macro_f1={r['f1_mac']:.4f}")

print("\n" + "="*65)
print(f"{'Method':<18} {'Acc':>7} {'MacroAUC':>10} {'WtdAUC':>9} {'MacroF1':>9}")
print("-"*65)
for m in results:
    r = results[m]
    print(f"{m:<18} {r['acc']:>7.4f} {r['auc_mac']:>10.4f} {r['auc_wtd']:>9.4f} {r['f1_mac']:>9.4f}")
print("="*65)

## 7. Comparison Plots

In [ ]:
methods       = list(results.keys())
method_colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]

# ── Plot 1: Method-level bar chart
accs     = [results[m]["acc"]     for m in methods]
auc_macs = [results[m]["auc_mac"] for m in methods]
auc_wtds = [results[m]["auc_wtd"] for m in methods]
x = np.arange(len(methods)); w = 0.25

fig, ax = plt.subplots(figsize=(11,5))
for i,(vals,lbl,alpha) in enumerate([
    (accs,"Accuracy",1.0),(auc_macs,"Macro AUC (OvR)",0.7),(auc_wtds,"Weighted AUC",0.45)]):
    bars = ax.bar(x+(i-1)*w, vals, w, label=lbl,
                  color=method_colors, alpha=alpha, edgecolor="white")
    for bar in bars:
        h=bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.003, f"{h:.3f}",
                ha="center", va="bottom", fontsize=7, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=11)
ax.set_ylim(0.6,1.02); ax.set_ylabel("Score",fontsize=12)
ax.set_title("DINOv2 (28×28 upscaled) — Method-Level Performance\nDermaMNIST Test Set",fontsize=13)
ax.legend(fontsize=10); ax.grid(axis="y",alpha=0.3)
plt.tight_layout()
plt.savefig("/content/28_method_acc_auc.png",dpi=150,bbox_inches="tight"); plt.show()

# ── Plot 2: Per-class AUC heatmap
auc_mat = np.array([results[m]["per_auc"] for m in methods])
fig, ax = plt.subplots(figsize=(13,4))
im = ax.imshow(auc_mat, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=1.0)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES,rotation=35,ha="right",fontsize=9)
ax.set_yticks(range(len(methods))); ax.set_yticklabels(methods,fontsize=10)
for i in range(len(methods)):
    for j in range(N_CLASSES):
        v=auc_mat[i,j]
        ax.text(j,i,f"{v:.3f}",ha="center",va="center",fontsize=9,fontweight="bold",
                color="black" if 0.6<v<0.9 else "white")
plt.colorbar(im,ax=ax,label="AUC (OvR)")
ax.set_title("Per-Class AUC (OvR) — 28×28",fontsize=13)
plt.tight_layout()
plt.savefig("/content/28_perclass_auc_heatmap.png",dpi=150,bbox_inches="tight"); plt.show()

# ── Plot 3: Per-class Recall heatmap
rec_mat = np.array([results[m]["per_rec"] for m in methods])
fig, ax = plt.subplots(figsize=(13,4))
im = ax.imshow(rec_mat, aspect="auto", cmap="RdYlGn", vmin=0.0, vmax=1.0)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES,rotation=35,ha="right",fontsize=9)
ax.set_yticks(range(len(methods))); ax.set_yticklabels(methods,fontsize=10)
for i in range(len(methods)):
    for j in range(N_CLASSES):
        v=rec_mat[i,j]
        ax.text(j,i,f"{v:.3f}",ha="center",va="center",fontsize=9,fontweight="bold",
                color="black" if 0.2<v<0.8 else "white")
plt.colorbar(im,ax=ax,label="Recall")
ax.set_title("Per-Class Recall — 28×28",fontsize=13)
plt.tight_layout()
plt.savefig("/content/28_perclass_recall_heatmap.png",dpi=150,bbox_inches="tight"); plt.show()

# ── Plot 4: Grouped bar (AUC + Recall)
fig, axes = plt.subplots(1,2,figsize=(16,5))
x=np.arange(N_CLASSES); n=len(methods); w=0.15
for ax,(mat,ylabel,title) in zip(axes,[
    (auc_mat,"AUC (OvR)","Per-Class AUC by Method"),
    (rec_mat,"Recall",   "Per-Class Recall by Method")]):
    for i,(mname,col) in enumerate(zip(methods,method_colors)):
        ax.bar(x+(i-n/2+0.5)*w, mat[i], w, label=mname, color=col, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(CLASS_SHORT,fontsize=10)
    ax.set_ylabel(ylabel,fontsize=11); ax.set_title(title,fontsize=12)
    ax.legend(fontsize=8,loc="lower right"); ax.set_ylim(0,1.15); ax.grid(axis="y",alpha=0.3)
fig.suptitle("DINOv2 (28×28 upscaled): Per-Class Performance by Method",fontsize=13)
plt.tight_layout()
plt.savefig("/content/28_perclass_grouped.png",dpi=150,bbox_inches="tight"); plt.show()

print("All plots saved to /content/")

## 8. Download Results

In [ ]:
from google.colab import files
import zipfile, os

with zipfile.ZipFile("/content/dermamnist_28_results.zip","w") as z:
    for f in Path("/content").glob("28_*.png"):
        z.write(f, f.name)
    for f in CKPT_DIR.glob("*.pt"):
        z.write(f, f"checkpoints/{f.name}")

files.download("/content/dermamnist_28_results.zip")
print("Downloaded: dermamnist_28_results.zip")